# WB — TFT + N-HiTS×5 + LGBM (с чекпоинтами TFT

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'neuralforecast'], check=True)
print('OK')

In [ ]:
import warnings, os, psutil, gc, glob
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, TFT
from neuralforecast.losses.pytorch import MAE

print(f'torch={torch.__version__} | cuda={torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

TARGET_COL = 'target_1h'
FP = 8
TRAIN_PATH = 'data/train_solo_track.parquet'
TEST_PATH  = 'data/test_solo_track.parquet'
PREDS_DIR  = 'data/preds_v6'
CKPT_HLD  = 'add your path here/ckpt_tft_hld'
CKPT_FULL = 'add your path here/ckpt_tft_full'
os.makedirs(CKPT_HLD,  exist_ok=True)
os.makedirs(CKPT_FULL, exist_ok=True)
print('OK')

## Monkey patch — обязателен

In [ ]:
_orig_init = pl.Trainer.__init__

def _patched_init(self, *args, **kwargs):
    forbidden = {
        'lstm_layers','num_heads','attn_dropout','hidden_size','dropout',
        'futr_exog_list','learning_rate','max_steps','batch_size',
        'random_seed','enable_progress_bar','trainer_kwargs'
    }
    for k in forbidden:
        kwargs.pop(k, None)
    _orig_init(self, *args, **kwargs)

pl.Trainer.__init__ = _patched_init
print('Monkey patch применён.')

## 1. Данные

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
test_df['timestamp']  = pd.to_datetime(test_df['timestamp'])
train_df = train_df.sort_values(['route_id','timestamp']).reset_index(drop=True)
test_df  = test_df.sort_values(['route_id','timestamp']).reset_index(drop=True)

INFERENCE_TS = train_df['timestamp'].max()
test_work = test_df.copy()
test_work['step_num'] = (
    (test_work['timestamp'] - INFERENCE_TS).dt.total_seconds() / 1800
).round().astype(int)

route_medians = train_df.groupby('route_id')[TARGET_COL].median().clip(lower=1.0)
print(f'INFERENCE_TS: {INFERENCE_TS}')
print(f'route_medians: min={route_medians.min():.0f} median={route_medians.median():.0f}')

In [ ]:
FUTURE_COVS = ['hour_norm','minute_norm','dow_norm','is_saturday','time_slot_norm']

def add_time_feats(df):
    df = df.copy()
    df['hour_norm']      = (df['ds'].dt.hour / 24.0).astype('float32')
    df['minute_norm']    = (df['ds'].dt.minute / 60.0).astype('float32')
    df['dow_norm']       = (df['ds'].dt.dayofweek / 6.0).astype('float32')
    df['is_saturday']    = (df['ds'].dt.dayofweek == 5).astype('float32')
    df['time_slot_norm'] = ((df['ds'].dt.hour*2 + df['ds'].dt.minute//30) / 47.0).astype('float32')
    return df

df_base = train_df[['route_id','timestamp',TARGET_COL]].copy()
df_base.columns = ['unique_id','ds','y']
df_base['y'] = np.log1p(df_base['y'] / df_base['unique_id'].map(route_medians))
df_tft = add_time_feats(df_base.copy())
HOLDOUT_START = pd.Timestamp('2025-10-25 11:00:00')
HOLDOUT_END   = pd.Timestamp('2025-10-25 14:30:00')
df_base_fit = df_base[df_base['ds'] < HOLDOUT_START].copy()
df_tft_fit  = df_tft[df_tft['ds']   < HOLDOUT_START].copy()
df_hld      = df_base[(df_base['ds'] >= HOLDOUT_START) & (df_base['ds'] <= HOLDOUT_END)].copy()
df_hld['y_true']   = np.expm1(df_hld['y']) * df_hld['unique_id'].map(route_medians)
df_hld['step_num'] = df_hld.groupby('unique_id').cumcount() + 1

future_tft = add_time_feats(
    test_df[['route_id','timestamp']].rename(columns={'route_id':'unique_id','timestamp':'ds'})
)
holdout_ts  = pd.date_range(HOLDOUT_START, HOLDOUT_END, freq='30T')
future_hld  = add_time_feats(pd.DataFrame({
    'unique_id': np.repeat(df_base['unique_id'].unique(), len(holdout_ts)),
    'ds':        np.tile(holdout_ts, df_base['unique_id'].nunique())
}))

print(f'df_base_fit: {df_base_fit.shape} | df_tft_fit: {df_tft_fit.shape}')
print(f'Holdout: {df_hld.shape} | y_true mean={df_hld["y_true"].mean():.0f}')

## 2. Метрика

In [ ]:
class WapePlusRbias:
    def calculate(self, y_true, y_pred):
        yt = np.asarray(y_true,'float64').ravel()
        yp = np.asarray(y_pred,'float64').ravel()
        return float(np.abs(yp-yt).sum()/(yt.sum()+1e-9)+abs(yp.sum()/(yt.sum()+1e-9)-1))
    def components(self, y_true, y_pred):
        yt = np.asarray(y_true,'float64').ravel()
        yp = np.asarray(y_pred,'float64').ravel()
        return np.abs(yp-yt).sum()/(yt.sum()+1e-9), yp.sum()/(yt.sum()+1e-9)-1

metric = WapePlusRbias()

def eval_raw(raw_df, col, tag):
    df = raw_df.copy()
    df['pred'] = np.expm1(df[col]) * df['unique_id'].map(route_medians)
    df['pred'] = df['pred'].clip(0, None)
    ev = df[['unique_id','ds','pred']].merge(df_hld[['unique_id','ds','y_true']], on=['unique_id','ds'])
    ev['step'] = ev.groupby('unique_id').cumcount() + 1
    sc = metric.calculate(ev['y_true'].values, ev['pred'].values)
    w, rb = metric.components(ev['y_true'].values, ev['pred'].values)
    steps = [metric.calculate(ev.loc[ev['step']==s,'y_true'], ev.loc[ev['step']==s,'pred'])
             for s in range(1, FP+1)]
    print(f'[{tag}] {sc:.4f} | WAPE={w:.4f} | RBias={rb:+.4f}')
    print(f'  steps: {" ".join(f"{x:.3f}" for x in steps)}')
    return sc, ev

print('OK')

## 3. TFT holdout — с чекпоинтами

In [ ]:
%%time

def make_tft(seed=42, max_steps=500, ckpt_dir=None):
    """
    Проверенные параметры для T4 16GB:
    - input_size=96  (48ч lookback)
    - hidden_size=32 (не вылетает ОOM)
    - batch_size=8   (стабильно)
    - precision='16-mixed' (-40% VRAM)
    - devices=1      (НЕ 2 — CUDA fork конфликт)
    - futr_df ТОЛЬКО в predict(), НЕ в fit()
    """
    callbacks = [
        EarlyStopping(monitor='valid_loss', patience=10, mode='min', verbose=True),
    ]
    if ckpt_dir:
        callbacks.append(ModelCheckpoint(
            dirpath=ckpt_dir,
            filename='step{step:05d}-loss{valid_loss:.4f}',
            every_n_train_steps=50,
            save_top_k=-1, 
            save_last=True,  
            monitor='valid_loss',
            mode='min',
        ))

    return TFT(
        h=FP,
        input_size=96,
        hidden_size=32,
        lstm_layers=2,
        num_heads=4,
        attn_dropout=0.1,
        dropout=0.1,
        futr_exog_list=FUTURE_COVS,
        loss=MAE(),
        learning_rate=1e-3,
        max_steps=max_steps,
        batch_size=8,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        random_seed=seed,
        enable_progress_bar=True,
        trainer_kwargs={
            'precision': '16-mixed',
            'callbacks': callbacks,
        }
    )


def find_ckpt(ckpt_dir):
    """Находит последний сохранённый чекпоинт по времени изменения."""
    ckpts = glob.glob(os.path.join(ckpt_dir, '*.ckpt'))
    if not ckpts:
        return None
    return max(ckpts, key=os.path.getmtime)

ckpt = find_ckpt(CKPT_HLD)

if ckpt:
    print(f'Чекпоинт найден: {os.path.basename(ckpt)}')
    print('   Загружаем модель, обучение пропускаем...')
    nf_tft_hld = NeuralForecast.load(path=CKPT_HLD)
    print('   Загружено!')
else:
    print('Чекпоинт не найден → обучаем с нуля...')
    nf_tft_hld = NeuralForecast(models=[make_tft(seed=42, max_steps=500, ckpt_dir=CKPT_HLD)], freq='30T')
    nf_tft_hld.fit(df=df_tft_fit, val_size=FP) 
    nf_tft_hld.save(path=CKPT_HLD, overwrite=True)
    print(f'Сохранено: {CKPT_HLD}')

print('TFT holdout готова!')
print('Чекпоинты:', sorted(os.listdir(CKPT_HLD)))

In [ ]:
raw_tft_hld = nf_tft_hld.predict(futr_df=future_hld)
sc_tft, ev_tft = eval_raw(raw_tft_hld, 'TFT', 'TFT-holdout')

## 4. N-HiTS × 5 seeds (не трогать — работает)

In [ ]:
def make_nhits(seed):
    return NHITS(
        h=FP, input_size=672,
        n_blocks=[1,1,1],
        mlp_units=[[256,256],[256,256],[256,256]],
        n_pool_kernel_size=[16,4,1],
        n_freq_downsample=[48,8,1],
        loss=MAE(), learning_rate=5e-4, max_steps=1500,
        batch_size=16, dropout_prob_theta=0.15,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1, random_seed=seed, enable_progress_bar=True,
    )

nhits_hld_preds, nhits_hld_scores = [], []

for seed in [42, 43, 44, 45, 46]:
    print(f'\n=== N-HiTS seed={seed} ===')
    nf = NeuralForecast(models=[make_nhits(seed)], freq='30T')
    nf.fit(df=df_base_fit, val_size=FP)
    raw = nf.predict(df=df_base_fit)
    sc, ev = eval_raw(raw, 'NHITS', f'NHITS-s{seed}')
    nhits_hld_preds.append(ev[['unique_id','ds','pred']].rename(columns={'pred':f'pred_s{seed}'}))
    nhits_hld_scores.append(sc)
    del nf; gc.collect()

nhits_ens_hld = nhits_hld_preds[0].copy()
for p in nhits_hld_preds[1:]:
    nhits_ens_hld = nhits_ens_hld.merge(p, on=['unique_id','ds'])
pred_cols = [c for c in nhits_ens_hld.columns if c.startswith('pred_')]
nhits_ens_hld['pred'] = nhits_ens_hld[pred_cols].mean(axis=1)

ev_ens = nhits_ens_hld.merge(df_hld[['unique_id','ds','y_true']], on=['unique_id','ds'])
ev_ens['step'] = ev_ens.groupby('unique_id').cumcount() + 1
sc_ens = metric.calculate(ev_ens['y_true'].values, ev_ens['pred'].values)
w_ens, rb_ens = metric.components(ev_ens['y_true'].values, ev_ens['pred'].values)
steps_ens = [metric.calculate(ev_ens.loc[ev_ens['step']==s,'y_true'],
                               ev_ens.loc[ev_ens['step']==s,'pred'])
             for s in range(1, FP+1)]
print(f'\nN-HiTS seeds: {[f"{s:.4f}" for s in nhits_hld_scores]}')
print(f'N-HiTS ансамбль: {sc_ens:.4f} | WAPE={w_ens:.4f} | RBias={rb_ens:+.4f}')
print(f'  steps: {" ".join(f"{x:.3f}" for x in steps_ens)}')

## 5. LGBM holdout предсказания

In [ ]:
val_lgbm, tst_lgbm, val_xgb, tst_xgb, y_val_s = {},{},{},{},{}
for step in range(1, FP+1):
    val_lgbm[step] = np.load(f'{PREDS_DIR}/lgbm_val_{step}.npy')
    tst_lgbm[step] = np.load(f'{PREDS_DIR}/lgbm_tst_{step}.npy')
    val_xgb[step]  = np.load(f'{PREDS_DIR}/xgb_val_{step}.npy')
    tst_xgb[step]  = np.load(f'{PREDS_DIR}/xgb_tst_{step}.npy')
    y_val_s[step]  = np.load(f'{PREDS_DIR}/y_val_{step}.npy')
tst_bst = {s: (tst_lgbm[s]+tst_xgb[s])/2 for s in range(1, FP+1)}

train_ts = pd.read_parquet(TRAIN_PATH, columns=['route_id','timestamp'])
train_ts['timestamp'] = pd.to_datetime(train_ts['timestamp'])
val_ts = train_ts[
    train_ts['timestamp'] >= train_ts['timestamp'].max() - pd.Timedelta(days=7)
].sort_values(['route_id','timestamp']).reset_index(drop=True)
hld_mask = (val_ts['timestamp'] >= HOLDOUT_START) & (val_ts['timestamp'] <= HOLDOUT_END)
hld_idx  = np.where(hld_mask.values)[0]

bst_hld = {}
for step in range(1, FP+1):
    vl = val_lgbm[step]; vx = val_xgb[step]
    n = min(len(hld_idx), len(vl), len(vx))
    bst_hld[step] = (vl[hld_idx[:n]] + vx[hld_idx[:n]]) / 2

print('LGBM holdout OK')
print(f'step1 mean: {bst_hld[1].mean():.0f}')

## 6. Подбор каскада на holdout

In [ ]:
def cascade_holdout(step_source):
    all_preds, all_true = [], []
    for step in range(1, FP+1):
        mask = ev_ens['step'] == step
        yt   = ev_ens.loc[mask, 'y_true'].values
        src  = step_source[step]
        if src == 'lgbm':
            n  = min(len(bst_hld[step]), len(yt))
            yp = bst_hld[step][:n] if n == len(yt) else np.full(len(yt), bst_hld[step].mean())
        elif src == 'tft':
            yp = ev_tft.loc[ev_tft['step']==step,'pred'].values[:len(yt)]
        else:
            yp = ev_ens.loc[mask,'pred'].values
        all_preds.append(yp); all_true.append(yt)
    return metric.calculate(np.concatenate(all_true), np.concatenate(all_preds))

strategies = {
    'NHiTS only':                    {s:'nhits' for s in range(1,9)},
    'TFT only':                      {s:'tft'   for s in range(1,9)},
    'LGBM[1]+NHiTS[2-8]':           {1:'lgbm', **{s:'nhits' for s in range(2,9)}},
    'LGBM[1-2]+NHiTS[3-8]':         {**{s:'lgbm' for s in range(1,3)}, **{s:'nhits' for s in range(3,9)}},
    'LGBM[1]+TFT[2-5]+NHiTS[6-8]':  {1:'lgbm',2:'tft',3:'tft',4:'tft',5:'tft',6:'nhits',7:'nhits',8:'nhits'},
    'LGBM[1-2]+TFT[3-5]+NHiTS[6-8]':{1:'lgbm',2:'lgbm',3:'tft',4:'tft',5:'tft',6:'nhits',7:'nhits',8:'nhits'},
    'TFT[1-5]+NHiTS[6-8]':          {**{s:'tft' for s in range(1,6)},**{s:'nhits' for s in range(6,9)}},
    'LGBM[1]+TFT[2-8]':             {1:'lgbm',**{s:'tft' for s in range(2,9)}},
}

per_step = {}
for step in range(1, FP+1):
    b_src, b_sc = 'nhits', 999
    for src in ['lgbm','tft','nhits']:
        sc2 = cascade_holdout({**{s:'nhits' for s in range(1,9)}, step:src})
        if sc2 < b_sc: b_sc, b_src = sc2, src
    per_step[step] = b_src
strategies['Best per step'] = per_step

print(f'NHiTS ens: {sc_ens:.4f} | TFT: {sc_tft:.4f}\n')
results = {}
for name, strat in strategies.items():
    sc2 = cascade_holdout(strat)
    results[name] = sc2
    print(f'  {name:45s}: {sc2:.4f}')

best_name  = min(results, key=results.get)
best_strat = strategies[best_name]
print(f'\n🏆 {best_name} = {results[best_name]:.4f}')
print(f'   По шагам: {best_strat}')

## 7. TFT финальная — с чекпоинтами

In [ ]:
%%time

ckpt_f = find_ckpt(CKPT_FULL)

if ckpt_f:
    print(f'Чекпоинт финальной TFT: {os.path.basename(ckpt_f)}')
    nf_tft_final = NeuralForecast.load(path=CKPT_FULL)
    print('   Загружено!')
else:
    print('Обучаем финальную TFT...')
    nf_tft_final = NeuralForecast(
        models=[make_tft(seed=42, max_steps=500, ckpt_dir=CKPT_FULL)], freq='30T')
    nf_tft_final.fit(df=df_tft, val_size=FP)
    nf_tft_final.save(path=CKPT_FULL, overwrite=True)
    print(f'Сохранено: {CKPT_FULL}')

raw_tft_tst = nf_tft_final.predict(futr_df=future_tft)
raw_tft_tst['pred'] = np.expm1(raw_tft_tst['TFT']) * raw_tft_tst['unique_id'].map(route_medians)
raw_tft_tst['pred'] = raw_tft_tst['pred'].clip(0, None)
raw_tft_tst['step_num'] = raw_tft_tst.groupby('unique_id').cumcount() + 1
raw_tft_tst = raw_tft_tst.rename(columns={'unique_id':'route_id'})
tft_map = raw_tft_tst.set_index(['route_id','step_num'])['pred'].to_dict()
print(f'TFT test: mean={raw_tft_tst["pred"].mean():.0f}')
print(f'Чекпоинты holdout:  {sorted(os.listdir(CKPT_HLD))}')
print(f'Чекпоинты финальные: {sorted(os.listdir(CKPT_FULL))}')

In [ ]:
%%time
print('=== N-HiTS × 5 seeds финальное ===')
nhits_tst_preds = []

for seed in [42, 43, 44, 45, 46]:
    print(f'seed={seed}...')
    nf = NeuralForecast(models=[make_nhits(seed)], freq='30T')
    nf.fit(df=df_base)
    raw = nf.predict(df=df_base)
    raw['pred'] = np.expm1(raw['NHITS']) * raw['unique_id'].map(route_medians)
    raw['pred'] = raw['pred'].clip(0, None)
    raw['step_num'] = raw.groupby('unique_id').cumcount() + 1
    raw = raw.rename(columns={'unique_id':'route_id'})
    nhits_tst_preds.append(raw[['route_id','step_num','pred']].rename(columns={'pred':f'pred_s{seed}'}))
    del nf; gc.collect()

nhits_tst_ens = nhits_tst_preds[0].copy()
for p in nhits_tst_preds[1:]:
    nhits_tst_ens = nhits_tst_ens.merge(p, on=['route_id','step_num'])
pred_cols = [c for c in nhits_tst_ens.columns if c.startswith('pred_')]
nhits_tst_ens['pred_ens'] = nhits_tst_ens[pred_cols].mean(axis=1)
nhits_map = nhits_tst_ens.set_index(['route_id','step_num'])['pred_ens'].to_dict()

print(f'NHiTS ens test: mean={nhits_tst_ens["pred_ens"].mean():.0f}')

## 8. Сабмиты

In [ ]:
NHITS_CAL = 1.010
best_c_tft, best_sc_tft = 1.0, sc_tft
for c in np.arange(0.90, 1.11, 0.005):
    sc_c = metric.calculate(ev_tft['y_true'].values, ev_tft['pred'].values * c)
    if sc_c < best_sc_tft: best_c_tft, best_sc_tft = c, sc_c
print(f'TFT cal: {sc_tft:.4f} → {best_sc_tft:.4f} (c={best_c_tft:.3f})')

def make_sub(arr, name):
    sub = pd.DataFrame({'id':test_work['id'].values,
                        'y_pred':np.clip(arr,0,None)}).sort_values('id').reset_index(drop=True)
    assert len(sub)==8000
    sub.to_csv(f'/kaggle/working/submission_solo_{name}.csv', index=False)
    print(f'{name}: mean={sub["y_pred"].mean():.0f}')
    return sub

def build(strat, nhits_sc=1.0, tft_sc=1.0):
    final = np.zeros(len(test_work))
    for i in range(len(test_work)):
        step  = int(test_work.iloc[i]['step_num'])
        route = int(test_work.iloc[i]['route_id'])
        src = strat[step]
        if src == 'lgbm':   final[i] = tst_bst[step][i]
        elif src == 'tft':  final[i] = tft_map.get((route,step),0) * tft_sc
        else:               final[i] = nhits_map.get((route,step),0) * nhits_sc
    return final

def build_w(wl, wt, wn, nhits_sc=1.0):
    final = np.zeros(len(test_work))
    for i in range(len(test_work)):
        step  = int(test_work.iloc[i]['step_num'])
        route = int(test_work.iloc[i]['route_id'])
        final[i] = (wl[step]*tst_bst[step][i] +
                    wt[step]*tft_map.get((route,step),0)*best_c_tft +
                    wn[step]*nhits_map.get((route,step),0)*nhits_sc)
    return final

sub1 = make_sub(build({s:'nhits' for s in range(1,9)}, NHITS_CAL), 'nhits_ens5')
sub2 = make_sub(build({s:'tft' for s in range(1,9)}, tft_sc=best_c_tft), 'tft_cal')
sub3 = make_sub(build(best_strat, NHITS_CAL, best_c_tft), 'cascade_best')
sub4 = make_sub(build({1:'lgbm',2:'tft',3:'tft',4:'tft',5:'tft',6:'nhits',7:'nhits',8:'nhits'},
                      NHITS_CAL, best_c_tft), 'lgbm1_tft25_nhits68')

sub5 = make_sub(build({1:'lgbm',2:'lgbm',**{s:'nhits' for s in range(3,9)}}, NHITS_CAL),
                'lgbm12_nhits38')

WL = {1:1.0,2:0.5,3:0.0,4:0.0,5:0.0,6:0.0,7:0.0,8:0.0}
WT = {1:0.0,2:0.3,3:0.6,4:0.6,5:0.4,6:0.2,7:0.0,8:0.0}
WN = {s:1.0-WL[s]-WT[s] for s in range(1,9)}
sub6 = make_sub(build_w(WL, WT, WN, NHITS_CAL), 'cascade_soft')

print(f'\n{"━"*55}')
print('HOLDOUT:')
print(f'  N-HiTS ×5:    {sc_ens:.4f}')
print(f'  TFT:          {sc_tft:.4f} → cal {best_sc_tft:.4f} (c={best_c_tft:.3f})')
print(f'  Best cascade: {results[best_name]:.4f}  ({best_name})')
print(f'{"━"*55}')
print('Загружайте: cascade_best → lgbm1_tft25_nhits68 → nhits_ens5 → tft_cal')
print(f'RAM: {psutil.Process(os.getpid()).memory_info().rss/1e9:.2f} GB')

In [ ]:
for f in sorted(os.listdir('/kaggle/working')):
    if f.endswith('.csv'):
        sz = os.path.getsize(f'/kaggle/working/{f}')//1024
        print(f'  {f}  {sz} KB')
print()
for d, label in [(CKPT_HLD,'holdout'), (CKPT_FULL,'финальная')]:
    print(f'TFT {label} чекпоинты ({d}):')
    for f in sorted(os.listdir(d)):
        sz = os.path.getsize(os.path.join(d,f))//1024
        print(f'  {f}  {sz} KB')